In [1]:
import os
os.chdir("../")
os.getcwd()

'/home/minh_khai/salinity/ai-agent-learning/quick-ai-agent-test'

In [2]:
import requests, os
from dotenv import load_dotenv
load_dotenv()

from tavily import TavilyClient
tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

In [3]:
from langchain.tools import tool

@tool
def web_search(query: str) -> str:
    """Search the web for information about a given query."""
    results = tavily.search(query, max_results=5)
    
    out = []
    for r in results['results']:
        out.append(
            f"Title: {r['title']}\nURL: {r['url']}\nSnippet: {r['content'][:300]}\n"
        )
    
    return "\n---\n".join(out)

In [4]:
output = web_search.invoke("What is the capital of Vietnam?")
print(output)

Title: TRAVEL to VIETNAM - How to Planning a trip to Vietnam. Visit Vietnam, Vietnam Trip Planner
URL: https://www.travelvietnam.com/travel-planning/what-is-the-capital-of-vietnam.html
Snippet: Hanoi: The Capital of Vietnam and a Must-Visit City. ## Hanoi: the capital of Vietnam and its history. Hanoi is the capital city of Vietnam, located in the northern part of the country, on the banks of the Red River. Hanoi is a city of culture, where you can enjoy the unique art, music, and cuisine 

---
Title: Vietnam's capital: The complete Hanoi travel guide in 2026 - Vinpearl
URL: https://vinpearl.com/en/hanoi-vietnam-capital-things-to-know
Snippet: Vietnam's capital city, Hanoi, is a top destination to visit in this country thanks to its rich culture and exquisite scenery.

---
Title: Hanoi is the capital city of Vietnam, located in the northern part of the ...
URL: https://www.instagram.com/reel/C4fDntsv3iN?hl=en
Snippet: Hanoi is the capital city of Vietnam, located in the northern part o

In [5]:
from bs4 import BeautifulSoup
from readability import Document
import trafilatura
import re

@tool
def scrape_url(url: str) -> str:
    """
    Scrape and extract clean readable content from a URL.
    Uses multiple extraction strategies for better reliability.
    """
    
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0 Safari/537.36"
        ),
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.google.com/",
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        html     = response.text
        
        # ------------------------------------------------
        # Strategy 1  ->  trafilatura (BEST for articles/blogs)
        # ------------------------------------------------
        extracted = trafilatura.extract(
            html,
            include_comments=False,
            include_tables=False
        )
        
        if extracted and len(extracted.strip()) > 200:
            return re.sub(r'\s+', ' ', extracted)[:5000]
        
        # ------------------------------------------------
        # Strategy 2  ->  readability
        # ------------------------------------------------
        doc         = Document(html)
        clean_html  = doc.summary()
        soup        = BeautifulSoup(clean_html, 'html.parser')
        
        for tag in soup(["script", "style", "nav", "footer", "header", "aside", "form"]):
            tag.decompose()
        
        text = soup.get_text(separator=" ", strip=True)
        if text and len(text.strip()) > 200:
            return re.sub(r'\s+', ' ', text)[:5000]
        
        # ------------------------------------------------
        # Strategy 3  ->  fallback full page extraction
        # ------------------------------------------------
        soup = BeautifulSoup(html, 'html.parser')
        for tag in soup(["script", "style", "nav", "footer", "header", "aside", "form"]):
            tag.decompose()
        
        text = soup.get_text(separator=" ", strip=True)
        return re.sub(r'\s+', ' ', text)[:5000] or "Could not extract meaningful content from the page."
        
    except requests.exceptions.Timeout:
        return "Request timed out while scraping the URL."

    except requests.exceptions.HTTPError as e:
        return f"HTTP error occurred: {str(e)}"

    except Exception as e:
        return f"Could not scrape URL: {str(e)}"

In [7]:
# Provided URLs from above
# https://www.travelvietnam.com/travel-planning/what-is-the-capital-of-vietnam.html
# https://vinpearl.com/en/hanoi-vietnam-capital-things-to-know
# https://www.instagram.com/reel/C4fDntsv3iN?hl=en
# https://www.quora.com/What-is-the-old-capital-of-Vietnam
# https://www.britannica.com/place/Hanoi

result = scrape_url.invoke("https://www.britannica.com/place/Hanoi")
result

"Hanoi News • - What is Hanoi? - Where is Hanoi located? - Why is Hanoi important to Vietnam? - What are some famous landmarks in Hanoi? - How has Hanoi's history shaped its culture? - What role does Hanoi play in Vietnam's government, education, and economy today? Hanoi, city, capital of Vietnam. The city is situated in northern Vietnam on the western bank of the Red River, about 85 miles (140 km) inland from the South China Sea. In addition to being the national capital, Hanoi is also a province-level municipality (thanh pho), administered by the central government. Area mun., 1,205 square miles (3,120 square km). Pop. (1999) city, 1,523,936; mun., 5,053,654; (2009) city, 2,316,722; mun., 6,451,909; (2014 est.) city, 3,292,000. History The region around present-day Hanoi was settled in prehistoric times, and the location was often chosen as a political centre by Chinese conquerors. In 1010 Ly Thai To, the first ruler of the Ly dynasty (1009–1225) of Vietnam, chose the site of Hanoi—t

In [8]:
# Tks to @tool, we can call .invoke()
# Tavily returns URLs of related articles
# But we have to manually pick one of them to feed in scrape_url() as above -> Not efficient

## AGENT

In [9]:
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
llm = ChatAnthropic(model="claude-haiku-4-5-20251001")

In [15]:
# 1st: First Agent: Search
def build_search_agent():
    return create_agent(
        model = llm,
        tools = [web_search],
    )

# 2nd: Second Agent: Render
def build_render_agent():
    return create_agent(
        model = llm,
        tools = [scrape_url],
    )

In [16]:
writer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert research writer. Write clear, structured and insightful reports."),
    ("human", """Write a detailed research report on the topic below.

Topic: {topic}

Research Gathered:
{research}

Structure the report as:
- Introduction
- Key Findings (minimum 3 well-explained points)
- Conclusion
- Sources (list all URLs found in the research)

Be detailed, factual and professional."""),
])

write_chain = writer_prompt | llm | StrOutputParser()

In [17]:
critic_prompt = ChatPromptTemplate.from_messages([
     ("system", "You are a sharp and constructive research critic. Be honest and specific."),
    ("human", """Review the research report below and evaluate it strictly.

Report:
{report}

Respond in this exact format:

Score: X/10

Strengths:
- ...
- ...

Areas to Improve:
- ...
- ...

One line verdict:
..."""),
])

critic_chain = critic_prompt | llm | StrOutputParser()

## Pipeline

In [22]:
def run_pipeline(topic: str, research = None) -> dict:
    state = {}
    
    # search agent working 
    print("\n"+" ="*50)
    print("step 1 - search agent is working ...")
    print("="*50)
    
    search_agent    = build_search_agent()
    search_results  = search_agent.invoke({
        "messages" : [(
            "user", f"Find recent, reliable and detailed information about: {topic}"
        )]
    })
    
    state["search_results"] = search_results['messages'][-1].content
    print("\n search result ",state['search_results'])
    
    # reader agent working
    print("\n"+" ="*50)
    print("step 2 - Reader agent is scraping top resources ...")
    print("="*50)
    
    render_agent  = build_render_agent()
    reader_result = render_agent.invoke({
        "messages" : [("user",
            f"Based on the following search results about '{topic}', "
            f"pick the most relevant URL and scrape it for deeper content.\n\n"
            f"Search Results:\n{state['search_results'][:800]}"
        )]
    })
    
    state['rendered_content'] = reader_result['messages'][-1].content
    print("\n rendered content ",state['rendered_content'])
    
    # writer chain
    print("\n"+" ="*50)
    print("step 3 - Writer is drafting the report ...")
    print("="*50)
    
    research_combined = (
        f"SEARCH RESULTS : \n {state['search_results']} \n\n"
        f"DETAILED SCRAPED CONTENT : \n {state['rendered_content']}"
    )
    
    state["report"] = write_chain.invoke({
        "topic" : topic,
        "research" : research_combined
    })
    
    print("\n Final Report\n",state['report'])


    #critic report 
    print("\n"+" ="*50)
    print("step 4 - critic is reviewing the report ")
    print("="*50)
    
    state["feedback"] = critic_chain.invoke({
        "report" : state['report']
    })
    
    print("\n critic report \n", state['feedback'])
    
    return state

In [23]:
topic = "The impact of AI on the job market in 2026"
run_pipeline(topic)


 = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = =
step 1 - search agent is working ...

 search result  Based on my search results, here's detailed, recent information about the impact of AI on the job market in 2026:

## Key Findings from 2026 Reports

### Job Displacement vs. Creation

**World Economic Forum's Future of Jobs Report 2026** provides the most comprehensive outlook:
- **92 million jobs will be displaced** by 2030
- **170 million new roles will be created** by 2030
- The net effect shows **job creation will outpace displacement** overall

### Major Shifts Already Occurring

1. **Reshaping Over Replacement**: BCG's analysis indicates that **AI will reshape more jobs than it replaces**. Rather than eliminating positions, many roles are being fundamentally transformed with new AI tools and workflows.

2. **Real 2025-2026 Trends**:
   - Over 54,000 jobs were cut in 2025 (notably including tech sector layoffs)
   - Meta's Mar

{'search_results': "Based on my search results, here's detailed, recent information about the impact of AI on the job market in 2026:\n\n## Key Findings from 2026 Reports\n\n### Job Displacement vs. Creation\n\n**World Economic Forum's Future of Jobs Report 2026** provides the most comprehensive outlook:\n- **92 million jobs will be displaced** by 2030\n- **170 million new roles will be created** by 2030\n- The net effect shows **job creation will outpace displacement** overall\n\n### Major Shifts Already Occurring\n\n1. **Reshaping Over Replacement**: BCG's analysis indicates that **AI will reshape more jobs than it replaces**. Rather than eliminating positions, many roles are being fundamentally transformed with new AI tools and workflows.\n\n2. **Real 2025-2026 Trends**:\n   - Over 54,000 jobs were cut in 2025 (notably including tech sector layoffs)\n   - Meta's March 2026 layoffs signaled measured AI-driven workforce displacement at scale\n   - Early-career hiring hit its lowest po